# 06 — Classification and retrieval robustness audit

Stream C, tasks **C6–C8**. This notebook reports classification performance by training-frequency group, clause length, and rule-detected party roles; tests one-factor counterfactual changes; evaluates illustrative European/French out-of-distribution examples; and compares dense retrieval rankings before and after the same perturbations.

It requires the classification test prediction files from Streams A/B and access to the selected final Legal-BERT checkpoint. Do not fill model-card metrics until this notebook has actually run. The OOD examples are synthetic probes without gold labels, so they support confidence/prediction inspection only—not accuracy claims.

## 1. Load the fixed test data and prediction files

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.audit.classifier import HuggingFaceTextPredictor
from src.audit.perturb import (
    audit_counterfactuals,
    audit_ood_examples,
    audit_retrieval_counterfactuals,
    generate_counterfactual_pairs,
    summarize_counterfactuals,
)
from src.audit.slices import build_audit_frame, slice_performance
from src.config import PREDICTIONS_DIR, RESULTS_DIR, ROOT, RUNS_DIR
from src.data import load_ledgar, to_pandas
from src.retrieval.dense import MODEL_NAME, build_dense_index
from src.utils import set_seed

set_seed(42)
dataset = load_ledgar()
train_df = to_pandas(dataset, "train")
test_df = to_pandas(dataset, "test")

# Change the run IDs if your teammates used different names.
FINAL_MODEL_RUN_ID = "legal-bert_final_20261019-1523"
TFIDF_RUN_ID = ""
final_prediction_path = PREDICTIONS_DIR / f"{FINAL_MODEL_RUN_ID}_test.parquet"
if not final_prediction_path.exists():
    raise FileNotFoundError(
        f"Missing final-model predictions: {final_prediction_path}. "
        "Copy the common-format test prediction file into results/predictions/ first."
    )
final_predictions = pd.read_parquet(final_prediction_path)

tfidf_predictions = None
if TFIDF_RUN_ID:
    tfidf_prediction_path = PREDICTIONS_DIR / f"{TFIDF_RUN_ID}_test.parquet"
    if not tfidf_prediction_path.exists():
        raise FileNotFoundError(f"Missing TF-IDF test predictions: {tfidf_prediction_path}")
    tfidf_predictions = pd.read_parquet(tfidf_prediction_path)

print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")
print(f"Final model predictions: {len(final_predictions):,}")
print("Year breakdown is not available from the shared loader; the test split is 2019 only.")

## 2. Per-slice classification performance (C6)

Head/middle/tail groups and short/medium/long boundaries are fitted using training data only. Slice macro-F1 averages classes present in that slice. Party roles are keyword-detected proxies, not contract metadata; categories may overlap. No within-test year comparison is possible because the fixed test split contains only 2019 and the shared loader omits row-level years.

In [ ]:
audit_frames = {}
slice_tables = []
prediction_sets = [("legal-bert-final", final_predictions)]
if tfidf_predictions is not None:
    prediction_sets.append(("tfidf-logreg", tfidf_predictions))
for model_name, predictions in prediction_sets:
    audit_frame = build_audit_frame(test_df, predictions, train_df)
    audit_frames[model_name] = audit_frame
    table = slice_performance(audit_frame)
    table.insert(0, "model", model_name)
    slice_tables.append(table)

slice_results = pd.concat(slice_tables, ignore_index=True)
display(slice_results)

## 3. Counterfactual classifier audit (C7–C8)

Pairs are generated only from test clauses containing a target feature, changing one feature per pair. These are heuristic stress tests and the transformed text should be reviewed before interpreting results. The model checkpoint path is recorded by the run config; on a local machine, copy the checkpoint from shared Drive or set `MODEL_CHECKPOINT_PATH` to its available location.

In [ ]:
run_config_path = RUNS_DIR / FINAL_MODEL_RUN_ID / "config.json"
run_config = json.loads(run_config_path.read_text())
MODEL_CHECKPOINT_PATH = run_config["best_checkpoint"]
if not Path(MODEL_CHECKPOINT_PATH).exists():
    raise FileNotFoundError(
        f"Final checkpoint is not available at {MODEL_CHECKPOINT_PATH}. "
        "Copy it from the shared Drive checkpoint folder and update MODEL_CHECKPOINT_PATH."
    )

predictor = HuggingFaceTextPredictor.from_pretrained(
    model_name=run_config["model_name"],
    checkpoint_path=MODEL_CHECKPOINT_PATH,
    revision=run_config.get("model_revision"),
    max_length=run_config["max_length"],
    batch_size=32,
)
counterfactual_pairs = generate_counterfactual_pairs(test_df)
print(f"Generated {len(counterfactual_pairs):,} one-feature counterfactual pairs")
counterfactual_results = audit_counterfactuals(counterfactual_pairs, predictor)
if not counterfactual_results.empty:
    expected_labels = final_predictions.set_index("idx")["pred"]
    saved_labels = counterfactual_results["idx"].map(expected_labels).astype(int)
    if not saved_labels.equals(counterfactual_results["original_label"]):
        raise ValueError("Checkpoint predictions do not match the saved final-model test predictions")
counterfactual_summary = summarize_counterfactuals(counterfactual_results)
display(counterfactual_summary)

## 4. European-style and French probes (C7–C8)

These authored synthetic examples have no gold labels. Report predictions and confidence only; do not calculate accuracy or describe these few examples as representative of all European or French contracts.

In [ ]:
ood_path = ROOT / "eval" / "ood" / "clauses.jsonl"
ood_examples = pd.DataFrame([json.loads(line) for line in ood_path.read_text().splitlines() if line.strip()])
ood_results = audit_ood_examples(ood_examples, predictor)
ood_summary = (
    ood_results.groupby(["language", "style"], as_index=False)
    .agg(
        n_examples=("id", "count"),
        mean_confidence=("predicted_confidence", "mean"),
        median_confidence=("predicted_confidence", "median"),
        distinct_predicted_classes=("predicted_label", "nunique"),
    )
)
display(ood_summary)
display(ood_results[["id", "pair_id", "language", "style", "predicted_label", "predicted_confidence", "text"]])

## 5. Retrieval stability under the same perturbations (C8)

Top-k Jaccard overlap and top-1 stability measure whether retrieval changes after a perturbation. These are stability indicators, not relevance or fairness scores.

In [ ]:
dense_retriever = build_dense_index(train_df, model_name=MODEL_NAME, batch_size=64)
retrieval_counterfactuals = audit_retrieval_counterfactuals(
    counterfactual_pairs,
    dense_retriever,
    k=5,
)
retrieval_summary = (
    retrieval_counterfactuals.groupby("category", as_index=False)
    .agg(
        n_pairs=("pair_id", "count"),
        top1_stability=("top1_same", "mean"),
        mean_top5_jaccard=("topk_jaccard", "mean"),
    )
)
display(retrieval_summary)

## 6. Save audit outputs

Review the generated counterfactual text before using aggregate results in the model card. Outputs contain measurements only; update `docs/model_card.md` after inspecting them and documenting the sample limitations.

In [ ]:
audit_results_dir = RESULTS_DIR / "audit"
audit_results_dir.mkdir(parents=True, exist_ok=True)
slice_results.to_csv(audit_results_dir / "slice_performance.csv", index=False)
pd.DataFrame(counterfactual_pairs).to_csv(audit_results_dir / "counterfactual_pairs.csv", index=False)
counterfactual_results.to_csv(audit_results_dir / "counterfactual_predictions.csv", index=False)
counterfactual_summary.to_csv(audit_results_dir / "counterfactual_summary.csv", index=False)
ood_results.to_csv(audit_results_dir / "ood_predictions.csv", index=False)
retrieval_counterfactuals.to_csv(audit_results_dir / "retrieval_counterfactuals.csv", index=False)
retrieval_summary.to_csv(audit_results_dir / "retrieval_robustness_summary.csv", index=False)
print(f"Saved audit tables under {audit_results_dir}")